<a href="https://colab.research.google.com/github/MalakMostafa-16003651/nyc-housing-transportation-analysis/blob/main/Person5_Research_Questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Person 5 (Kenzy) — Visualizations, Map & Story
**RQ9 — Where in NYC do high affordable-housing production and high serious-crash levels overlap?** *(map)*
**RQ10 — Do years with higher affordable-housing production also have higher or lower severe injuries?** *(line chart + change scatter)*

**Inputs (from the repository root):**
- `df_integrated.csv`: Person 3's ZIP × year table (housing + SIRTA crashes, matched ZIP-years only)
- `df_housing_clean.csv`: Person 1's cleaned housing data (used for building coordinates and citywide yearly totals)
- `df_transport_clean.csv`: Person 2's cleaned SIRTA crash data (used for crash coordinates and citywide yearly totals)

**Datasets:** [Affordable Housing Production by Building (hg8x-zxpr)](https://data.cityofnewyork.us/d/hg8x-zxpr) · [SIRTA serious crashes (xeqp-qz8h)](https://data.cityofnewyork.us/d/xeqp-qz8h)

### Setup and shared chart style
Every chart in my section uses Plotly (the same library as the Dash website), so they can be reused there without redrawing. Colours are the website's shared palette from `app/charts/style.py`: **blue `#2a6fdb` = affordable housing**, **red `#c43d5a` = crashes/injuries**, and the same five borough colours, so the notebook and website look identical. `style_fig()` gives every figure the same title position, font, legend and source note.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from IPython.display import display

# Team colours: same meaning -> same colour in every chart
# Same values as app/charts/style.py (the website), so notebook and website match
COLOR_HOUSING = "#2a6fdb"   # blue   = affordable housing
COLOR_CRASH   = "#c43d5a"   # red    = serious crashes / injuries
COLOR_BOTH    = "#6a3d9a"   # purple = high on both
COLOR_OTHER   = "#c9ced3"   # grey   = context
BOROUGH_COLORS = {"Manhattan": "#2a6fdb", "Brooklyn": "#e0702b", "Queens": "#2f9e6e",
                  "Bronx": "#c43d5a", "Staten Island": "#7d5bbe"}
SOURCE_NOTE = "Source: NYC Open Data — Affordable Housing Production by Building (hg8x-zxpr); SIRTA crashes (xeqp-qz8h)"

def style_fig(fig, title, x_title=None, y_title=None, height=520, source=True):
    """Apply the team's common look to any Plotly figure."""
    fig.update_layout(
        template="plotly_white",
        title=dict(text=title, x=0.01, xanchor="left", font=dict(size=18)),
        font=dict(family="Arial", size=13, color="#222"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=60, r=30, t=95, b=80 if source else 50),
        height=height,
        hoverlabel=dict(bgcolor="white", font_size=12),
    )
    if x_title is not None: fig.update_xaxes(title_text=x_title)
    if y_title is not None: fig.update_yaxes(title_text=y_title)
    if source:
        fig.add_annotation(text=SOURCE_NOTE, xref="paper", yref="paper", x=0, y=-0.17,
                           showarrow=False, xanchor="left", font=dict(size=10, color="#777"))
    return fig

print("Style ready")

Style ready


### Load the team's data
I always reload the three CSVs from the repository root (like Person 1's notebook) so I never use a stale DataFrame. In Colab, where the folder starts empty, the cell opens an upload button for any missing file instead of crashing. ZIP codes are stored as 5-character text so they join and display the same way everywhere (e.g. `10456`, not `10456.0`).

In [2]:
from pathlib import Path

NEEDED = ["df_integrated.csv", "df_housing_clean.csv", "df_transport_clean.csv"]
missing = [f for f in NEEDED if not Path(f).is_file()]

if missing:
    try:
        from google.colab import files          # only exists in Colab
        print("Please upload:", ", ".join(missing))
        files.upload()                           # opens a file picker; select all missing CSVs
    except ImportError:
        raise FileNotFoundError(f"Missing {missing} — run from the repository root.")

still_missing = [f for f in NEEDED if not Path(f).is_file()]
if still_missing:
    raise FileNotFoundError(f"Still missing {still_missing}. Upload them with the exact same names.")

df_integrated      = pd.read_csv("df_integrated.csv")
df_housing_clean   = pd.read_csv("df_housing_clean.csv", low_memory=False)
df_transport_clean = pd.read_csv("df_transport_clean.csv", low_memory=False)

def zip_text(s):
    """Numeric/float ZIPs -> '10456' strings; missing stays missing."""
    return pd.to_numeric(s, errors="coerce").astype("Int64").astype("string").str.zfill(5)

for d in (df_integrated, df_housing_clean, df_transport_clean):
    d["zip_code"] = zip_text(d["zip_code"])
    d["year"] = pd.to_numeric(d["year"], errors="coerce").astype("Int64")

print("df_integrated:     ", df_integrated.shape)
print("df_housing_clean:  ", df_housing_clean.shape)
print("df_transport_clean:", df_transport_clean.shape)
print("SIRTA crash dates: ", df_transport_clean["crash_date"].min(), "to", df_transport_clean["crash_date"].max())

df_integrated:      (484, 11)
df_housing_clean:   (9551, 42)
df_transport_clean: (8072, 26)
SIRTA crash dates:  2022-10-01 to 2026-06-30


## RQ9 — Where do high affordable housing and high serious crashes overlap?
**Method**
1. Total affordable units and serious crashes per ZIP over **all matched years (Oct 2022 – Jun 2026)**. This is the same window Person 1 used in RQ1, so our numbers agree, and every ZIP is measured over the same period.
2. A ZIP is **"high"** on a measure if it is in the **top 25 %** of ZIPs (≥ 75th percentile). A percentile adapts to the data; a median split was rejected because it would label half the city "high".
3. Each ZIP falls into one group: **High on both**, **High housing only**, **High crashes only**, or **Neither**.

In [3]:
zip_totals = (df_integrated.groupby("zip_code", as_index=False)
              .agg(borough=("borough", lambda s: s.mode().iat[0]),
                   affordable_units=("affordable_units", "sum"),
                   serious_crashes=("serious_crash_count", "sum"),
                   severe_injuries=("severe_injuries", "sum"),
                   fatalities=("fatalities", "sum"),
                   years_matched=("year", "nunique")))

HIGH_Q = 0.75
units_cut = zip_totals["affordable_units"].quantile(HIGH_Q)
crash_cut = zip_totals["serious_crashes"].quantile(HIGH_Q)
high_housing = zip_totals["affordable_units"] >= units_cut
high_crashes = zip_totals["serious_crashes"] >= crash_cut

GROUP_ORDER = ["High on both", "High housing only", "High crashes only", "Neither"]
zip_totals["group"] = np.select([high_housing & high_crashes, high_housing, high_crashes],
                                GROUP_ORDER[:3], default="Neither")

print(f"{len(zip_totals)} ZIP codes")
print(f"'High housing' = top 25% = at least {units_cut:,.0f} affordable units")
print(f"'High crashes' = top 25% = at least {crash_cut:,.0f} serious crashes")
display(zip_totals["group"].value_counts().reindex(GROUP_ORDER).to_frame("ZIP codes"))

143 ZIP codes
'High housing' = top 25% = at least 1,172 affordable units
'High crashes' = top 25% = at least 65 serious crashes


,ZIP codes
group,
High on both,23
High housing only,13
High crashes only,15
Neither,92


### Placing each ZIP on the map
`df_integrated` has no coordinates, so each ZIP's position is the **average location of its housing buildings and serious crashes** from the two cleaned datasets. Points outside the NYC bounding box are ignored. Rows with missing coordinates (≈ 16 % of SIRTA crashes) are skipped only for positioning; they are still counted in the totals. Using real event locations puts each circle where the activity actually is rather than at an arbitrary ZIP centre.

In [4]:
def nyc_points(df, lat="latitude", lon="longitude"):
    pts = df[["zip_code", lat, lon]].rename(columns={lat: "lat", lon: "lon"}).dropna()
    inside = pts["lat"].between(40.49, 40.92) & pts["lon"].between(-74.27, -73.68)
    return pts[inside]

housing_pts = nyc_points(df_housing_clean)
crash_pts   = nyc_points(df_transport_clean)
centroids = pd.concat([housing_pts, crash_pts]).groupby("zip_code")[["lat", "lon"]].mean()

zip_map = zip_totals.join(centroids, on="zip_code")
print("ZIPs without coordinates:", zip_map["lat"].isna().sum())
zip_map = zip_map.dropna(subset=["lat", "lon"])

ZIPs without coordinates: 0


### RQ9 map: housing and crash hotspots
- **Circles = ZIP codes.** Colour = group, size = affordable units. Hover over a circle for its numbers.
- Click **"Individual serious crashes"** or **"Affordable housing buildings"** in the legend to overlay the raw points from both datasets.
- Grey "Neither" ZIPs are drawn first so the coloured hotspots sit on top.

In [5]:
GROUP_COLORS = {"High on both": COLOR_BOTH, "High housing only": COLOR_HOUSING,
                "High crashes only": COLOR_CRASH, "Neither": COLOR_OTHER}
max_units = zip_map["affordable_units"].max()

fig_map = go.Figure()

# Optional raw layers (hidden until clicked in the legend)
fig_map.add_trace(go.Scattermap(
    lat=crash_pts["lat"], lon=crash_pts["lon"], mode="markers",
    marker=dict(size=4, color=COLOR_CRASH, opacity=0.35),
    name="Individual serious crashes", hoverinfo="skip", visible="legendonly"))
fig_map.add_trace(go.Scattermap(
    lat=housing_pts["lat"], lon=housing_pts["lon"], mode="markers",
    marker=dict(size=4, color=COLOR_HOUSING, opacity=0.35),
    name="Affordable housing buildings", hoverinfo="skip", visible="legendonly"))

# One layer per group, "Neither" first so hotspots are on top
for group in reversed(GROUP_ORDER):
    g = zip_map[zip_map["group"] == group]
    fig_map.add_trace(go.Scattermap(
        lat=g["lat"], lon=g["lon"], mode="markers", name=f"{group} ({len(g)})",
        marker=dict(size=8 + 30 * np.sqrt(g["affordable_units"] / max_units),   # sqrt: area ~ units
                    color=GROUP_COLORS[group], opacity=0.85),
        customdata=g[["zip_code", "borough", "affordable_units", "serious_crashes", "severe_injuries", "fatalities"]],
        hovertemplate=("<b>ZIP %{customdata[0]}</b> — %{customdata[1]}<br>"
                       "Affordable units: %{customdata[2]:,.0f}<br>"
                       "Serious crashes: %{customdata[3]:,.0f}<br>"
                       "Severe injuries: %{customdata[4]:,.0f}<br>"
                       "Fatalities: %{customdata[5]:,.0f}<extra></extra>")))

style_fig(fig_map, "Where affordable housing and serious crashes overlap — NYC ZIP codes, Oct 2022 – Jun 2026",
          height=680, source=False)
fig_map.update_layout(map=dict(style="carto-positron", center=dict(lat=40.71, lon=-73.93), zoom=9.4),
                      margin=dict(l=10, r=10, t=95, b=10),
                      legend=dict(title=dict(text="Top 25% of ZIPs on…  (circle size = affordable units)  ")))
fig_map.show()

### Which ZIPs overlap, and how much do they account for?
This table backs up the map with exact numbers and measures how concentrated the problem is.

In [6]:
overlap = (zip_totals[zip_totals["group"] == "High on both"]
           .sort_values("serious_crashes", ascending=False)
           .drop(columns="group").reset_index(drop=True))

share_zips   = len(overlap) / len(zip_totals)
share_units  = overlap["affordable_units"].sum() / zip_totals["affordable_units"].sum()
share_crash  = overlap["serious_crashes"].sum() / zip_totals["serious_crashes"].sum()
share_deaths = overlap["fatalities"].sum() / zip_totals["fatalities"].sum()

print(f"{len(overlap)} of {len(zip_totals)} ZIPs ({share_zips:.0%}) are high on BOTH measures.")
print("By borough:", overlap["borough"].value_counts().to_dict())
print(f"They hold {share_units:.0%} of affordable units, {share_crash:.0%} of serious crashes "
      f"and {share_deaths:.0%} of crash deaths.")
rho, p = stats.spearmanr(zip_totals["affordable_units"], zip_totals["serious_crashes"])
print(f"Spearman ρ across all ZIPs = {rho:.2f} (p = {p:.1e}) — matches Person 1's RQ1.")
display(overlap)

23 of 143 ZIPs (16%) are high on BOTH measures.
By borough: {'Bronx': 11, 'Brooklyn': 8, 'Queens': 2, 'Manhattan': 2}
They hold 44% of affordable units, 34% of serious crashes and 29% of crash deaths.
Spearman ρ across all ZIPs = 0.64 (p = 3.8e-18) — matches Person 1's RQ1.


,zip_code,borough,affordable_units,serious_crashes,severe_injuries,fatalities,years_matched
0,11207,Brooklyn,2267.0,128.0,125.0,9.0,5
1,11212,Brooklyn,2698.0,119.0,115.0,8.0,5
2,11203,Brooklyn,1349.0,118.0,123.0,6.0,5
3,10456,Bronx,2968.0,109.0,112.0,2.0,5
4,11206,Brooklyn,1405.0,101.0,92.0,9.0,5
5,10467,Bronx,2402.0,100.0,102.0,8.0,5
6,11101,Queens,1771.0,98.0,94.0,6.0,5
7,10457,Bronx,3419.0,95.0,104.0,3.0,5
8,11208,Brooklyn,3578.0,90.0,98.0,5.0,5
9,10468,Bronx,2577.0,88.0,87.0,7.0,5


### RQ9 — Finding
- **23 of 143 ZIP codes (16 %) are in the top 25 % for both** affordable housing (≥ 1,172 units) and serious crashes (≥ 65 crashes). Together they hold **44 % of the matched affordable units, 34 % of serious crashes and 29 % of crash deaths**.
- The overlap is **not spread across the city**. It forms **two clusters**: the **South/Central Bronx** (11 ZIPs, e.g. 10456, 10467, 10457, 10468, 10453) and **Central/East Brooklyn** (8 ZIPs, e.g. 11207, 11212, 11203, 11206, 11208). Only **2 Queens** ZIPs (11101 Long Island City, 11368 Corona) and **2 Manhattan** ZIPs (10036 Midtown West, 10035 East Harlem) qualify, and **none in Staten Island**.
- The worst ZIPs are **11207 (East New York, 128 serious crashes, 2,267 units)**, **11212 (Brownsville, 119 crashes, 2,698 units)** and **11203 (East Flatbush, 118 crashes)**.
- Some ZIPs break the pattern: **10452** (Bronx) has the most affordable units in the city (5,001) but is not a crash hotspot (43), while southeast Brooklyn ZIPs **11234** (103 crashes, 157 units) and **11236** (79 crashes, 5 units) are crash hotspots with almost no affordable housing.
- The map shows **where** the overlap is. It does **not** show that housing causes crashes. Both are concentrated in dense neighbourhoods crossed by wide arterial roads.

*(If you rerun with different data and the numbers change, update this text from the output above.)*

## RQ10 — Do years with higher affordable-housing production also have higher or lower severe injuries?
Person 2's RQ3 already compares the yearly totals on two axes, so RQ10 goes one step further and tests the question two ways:

1. **Citywide, full years only (line chart).** Only **2023, 2024 and 2025** are complete calendar years in both datasets. SIRTA starts on 1 Oct 2022 and both datasets stop at 30 Jun 2026, so 2022 and 2026 are left out instead of being compared as if complete. Both series are shown as an **index (2023 = 100)** so they share one axis without the distortion of a dual axis. Citywide totals come from the cleaned datasets rather than `df_integrated`, so ZIP-years without a match are not lost.
2. **Within each ZIP (change scatter).** Three yearly points are too few for a correlation, so I compare **each ZIP with itself**: change in affordable units vs change in severe injuries from 2023→2024 and 2024→2025. Comparing a ZIP with itself removes fixed differences in size, population and road network between ZIPs, which a plain ZIP-vs-ZIP comparison cannot do.

*Quarterly data was considered and rejected: most housing projects are logged as starting in late June or December (HPD reporting deadlines), so quarterly housing totals reflect paperwork timing rather than construction.*

In [7]:
FULL_YEARS = [2023, 2024, 2025]

yearly = pd.DataFrame({
    "affordable_units": df_housing_clean.groupby("year")["all_counted_units"].sum(),
    "severe_injuries":  df_transport_clean.groupby("year")["total_severe_injuries"].sum(),
    "serious_crashes":  df_transport_clean.groupby("year").size(),
    "fatalities":       df_transport_clean.groupby("year")["total_fatalities"].sum(),
}).loc[FULL_YEARS]
yearly.index = yearly.index.astype(int)
yearly.index.name = "year"

indexed = (yearly / yearly.loc[2023] * 100).round(1)
display(yearly.astype(int))
print("Change 2023 → 2025:")
print((yearly.loc[2025] / yearly.loc[2023] - 1).map("{:+.1%}".format).to_string())

,affordable_units,severe_injuries,serious_crashes,fatalities
year,,,,
2023,29113,2064,2055,135
2024,28965,2207,2180,147
2025,29277,2425,2367,155


Change 2023 → 2025:
affordable_units     +0.6%
severe_injuries     +17.5%
serious_crashes     +15.2%
fatalities          +14.8%


### RQ10 chart A: citywide trend, full years (2023 = 100)
Both series start at 100 in 2023, so they can share one axis even though housing is counted in thousands of units and injuries in hundreds. A line above 100 means higher than in 2023. The labels on the points show the real totals.

In [8]:
fig_trend = go.Figure()
# Housing labels go below the line and injury labels above, so they never overlap
series = [("affordable_units", "Affordable units started", COLOR_HOUSING, "bottom center"),
          ("severe_injuries",  "Severe crash injuries",    COLOR_CRASH,   "top center")]
for col, label, color, position in series:
    fig_trend.add_trace(go.Scatter(
        x=indexed.index, y=indexed[col], name=label, mode="lines+markers+text",
        line=dict(color=color, width=3), marker=dict(size=10),
        text=[f"{v:,.0f}" for v in yearly[col]], textposition=position,
        customdata=yearly[col],
        hovertemplate=f"{label}<br>%{{x}}: %{{customdata:,.0f}} (index %{{y:.1f}})<extra></extra>"))

fig_trend.add_hline(y=100, line_dash="dot", line_color="#999")
style_fig(fig_trend, "Severe injuries rose while affordable housing stayed flat — NYC, 2023–2025",
          x_title="Year", y_title="Index (2023 = 100)")
fig_trend.update_xaxes(dtick=1)
fig_trend.update_yaxes(range=[90, 125])
fig_trend.add_annotation(text="Labels show actual yearly totals. 2022 and 2026 are excluded (partial years).",
                         xref="paper", yref="paper", x=0, y=1.0, showarrow=False, xanchor="left",
                         font=dict(size=11, color="#555"))
fig_trend.show()

### RQ10 chart B: within-ZIP changes
Each dot is one ZIP code in one pair of years (2023→2024 or 2024→2025). Only ZIP-years matched in both years are used (`df_integrated`).
- **Right of 0**: the ZIP produced more affordable units than the year before.
- **Above 0**: it had more severe injuries than the year before.

If high-production years brought more injuries, the dots would slope upward.

In [9]:
wide = (df_integrated[df_integrated["year"].isin(FULL_YEARS)]
        .pivot_table(index=["zip_code", "borough"], columns="year",
                     values=["affordable_units", "severe_injuries"]))

pairs = []
for a, b in [(2023, 2024), (2024, 2025)]:
    pair = pd.DataFrame({
        "unit_change":   wide[("affordable_units", b)] - wide[("affordable_units", a)],
        "injury_change": wide[("severe_injuries", b)]  - wide[("severe_injuries", a)],
    }).dropna()                                   # ZIP must be matched in both years
    pair["years"] = f"{a}→{b}"
    pairs.append(pair)
changes = pd.concat(pairs).reset_index()

rho_c, p_rho_c = stats.spearmanr(changes["unit_change"], changes["injury_change"])
r_c, p_r_c     = stats.pearsonr(changes["unit_change"], changes["injury_change"])

fig_change = px.scatter(
    changes, x="unit_change", y="injury_change", color="years",
    color_discrete_sequence=["#93b7ee", "#1d4fa0"],   # light/dark housing blue = earlier/later pair
    hover_data={"zip_code": True, "borough": True, "years": True},
    labels={"unit_change": "Change in affordable units vs previous year",
            "injury_change": "Change in severe injuries vs previous year",
            "zip_code": "ZIP", "borough": "Borough", "years": "Years"})
# Straight-line fit over all dots (numpy, so no extra package is needed)
slope, intercept = np.polyfit(changes["unit_change"], changes["injury_change"], 1)
x_line = np.array([changes["unit_change"].min(), changes["unit_change"].max()])
fig_change.add_trace(go.Scatter(x=x_line, y=slope * x_line + intercept, mode="lines",
                                name="Trend (all dots)", line=dict(color="#444", dash="dash")))
fig_change.add_hline(y=0, line_color="#bbb"); fig_change.add_vline(x=0, line_color="#bbb")
fig_change.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color="white")),
                         selector=dict(mode="markers"))
style_fig(fig_change,
          f"More housing in a ZIP did not bring more injuries the same year (Spearman ρ = {rho_c:.2f}, p = {p_rho_c:.2f}, n = {len(changes)})",
          height=560)
fig_change.update_xaxes(tickformat=",", zeroline=False); fig_change.update_yaxes(zeroline=False)
fig_change.show()

print(f"Spearman ρ = {rho_c:+.2f} (p = {p_rho_c:.2f}); Pearson r = {r_c:+.2f} (p = {p_r_c:.2f}); n = {len(changes)} ZIP-year changes")
changes["housing_change"] = pd.cut(changes["unit_change"], [-np.inf, -100, 100, np.inf],
                                   labels=["Fell by >100 units", "Roughly flat (±100)", "Rose by >100 units"])
display(changes.groupby("housing_change", observed=True)["injury_change"]
        .agg(zip_years="size", avg_injury_change="mean", median_injury_change="median").round(2))

Spearman ρ = -0.09 (p = 0.24); Pearson r = -0.13 (p = 0.09); n = 167 ZIP-year changes


,zip_years,avg_injury_change,median_injury_change
housing_change,,,
Fell by >100 units,56,2.21,3.0
Roughly flat (±100),62,1.71,1.5
Rose by >100 units,49,1.55,1.0


### RQ10 — Finding
- **Citywide:** affordable-housing production was **almost perfectly flat** across the full years (29,113 → 28,965 → 29,277 units, **+0.6 %**), while **severe injuries rose every year** (2,064 → 2,207 → 2,425, **+17.5 %**) and crash deaths rose from 135 to 155. Production barely changed, so it cannot explain a 17 % increase in injuries.
- **Within ZIPs:** when a ZIP produced more affordable units than the year before, its severe injuries did **not** rise with it. The relationship is essentially zero and slightly negative (**Spearman ρ = −0.09, p = 0.24, n = 167**). ZIPs where production rose by more than 100 units had about the same injury change (+1.6 on average) as ZIPs where it fell (+2.2).
- **Answer:** No. Years with more affordable housing did **not** have more severe injuries. Injuries went up across the city regardless of how much housing was produced. The **places** with more housing also have more crashes (RQ9/RQ1), but year-to-year **changes** in housing do not move injuries. The spatial link most likely reflects shared neighbourhood traits (density, arterial roads), not the housing itself.
- **Caution:** only 3 complete years; housing is dated by **project start**, and residents move in years later, so any effect of new residents would appear after this window.

### Key numbers for the story
This cell reprints every number used in the findings and the story, so if the data is updated we can rerun it and check that the text above still matches the output.

In [10]:
# Key numbers for the story — rerun after any data change and update the markdown
print("=== RQ9 ===")
print(f"{len(overlap)}/{len(zip_totals)} ZIPs high on both | boroughs {overlap['borough'].value_counts().to_dict()}")
print(f"share of units {share_units:.0%}, crashes {share_crash:.0%}, deaths {share_deaths:.0%}")
print("Top 5:", ", ".join(f"{z} ({b})" for z, b in overlap[["zip_code", "borough"]].head(5).values))
print("\n=== RQ10 ===")
chg = yearly.loc[2025] / yearly.loc[2023] - 1
print(f"2023→2025: units {chg['affordable_units']:+.1%}, severe injuries {chg['severe_injuries']:+.1%}, deaths {chg['fatalities']:+.1%}")
print(f"Within-ZIP change correlation: ρ = {rho_c:+.2f}, p = {p_rho_c:.2f}, n = {len(changes)}")

=== RQ9 ===
23/143 ZIPs high on both | boroughs {'Bronx': 11, 'Brooklyn': 8, 'Queens': 2, 'Manhattan': 2}
share of units 44%, crashes 34%, deaths 29%
Top 5: 11207 (Brooklyn), 11212 (Brooklyn), 11203 (Brooklyn), 10456 (Bronx), 11206 (Brooklyn)

=== RQ10 ===
2023→2025: units +0.6%, severe injuries +17.5%, deaths +14.8%
Within-ZIP change correlation: ρ = -0.09, p = 0.24, n = 167


### Export for the website
Saves the three figures as standalone HTML and the tables behind them. It also writes **`data/zip_centroids.csv`**, the ZIP positions the website's RQ9 map (`app/charts/rq9_rq10.py`) reads, because `df_integrated` has no coordinates.

In [11]:
import os
os.makedirs("outputs", exist_ok=True)
for name, fig in {"rq9_overlap_map": fig_map, "rq10_trend": fig_trend, "rq10_zip_changes": fig_change}.items():
    fig.write_html(f"outputs/{name}.html", include_plotlyjs="cdn")
zip_map.to_csv("outputs/rq9_zip_overlap.csv", index=False)
yearly.to_csv("outputs/rq10_yearly_full_years.csv")
changes.to_csv("outputs/rq10_zip_changes.csv", index=False)
print("Saved:", sorted(os.listdir("outputs")))

# ZIP positions for the website map
os.makedirs("data", exist_ok=True)
centroids.round(6).reset_index().to_csv("data/zip_centroids.csv", index=False)
print("Saved data/zip_centroids.csv:", len(centroids), "ZIPs")

Saved: ['rq10_trend.html', 'rq10_yearly_full_years.csv', 'rq10_zip_changes.csv', 'rq10_zip_changes.html', 'rq9_overlap_map.html', 'rq9_zip_overlap.csv']
Saved data/zip_centroids.csv: 190 ZIPs


## Chart consistency review (all team charts)
I checked every chart in the project against this list:
| Rule | Why |
|---|---|
| Blue `#2a6fdb` = housing, red `#c43d5a` = crashes/injuries, everywhere (website `style.py`) | Readers learn the colours once |
| Same borough colours in every borough chart | Brooklyn shouldn't change colour between charts |
| Title says **what + where + when** (e.g. "…by ZIP code, NYC, Oct 2022 – Jun 2026") | Each chart stands alone on the website |
| Axis titles in plain English with units, not column names (`serious_crash_count` → "Serious crashes") | Avoids the "unclear labels" deduction |
| Partial years (2022 = Oct–Dec only, 2026 = Jan–Jun only) labelled or excluded from year comparisons | Prevents false "drops" in 2022 and 2026 |
| Thousands separators, years as whole numbers | Readability |
| Correlation charts show ρ/r, p and n | Honest strength of evidence |
| No chart claims causation in its title | Avoids the "unsupported causation" deduction |

**Fixes requested from teammates after the review:**
- **Person 1 (RQ1, RQ2, matplotlib):** colours `#2E86AB`/`#E4572E` differ slightly from the website's `#2a6fdb`/`#c43d5a`; swap them. Axis labels say "(all years)"; change to "(Oct 2022 – Jun 2026)", because 2022 and 2026 are partial.
- **Person 4 (RQ7, RQ8):** good. Already uses the website borough colours and states 2022–2026. RQ8 title could add "(Staten Island: only 15 projects)" since that bar is unreliable.
- **Person 2 (RQ3, website):** title says "boundary years may be partial"; be specific: "2022 = Oct–Dec only, 2026 = Jan–Jun only".
- **Person 3 (integration notebook):** the draft `rq1` category scatter duplicates RQ1/RQ9; label it as a draft or remove it.
- **Website:** all charts already share `apply_style()`; RQ9 and RQ10 now use it too.

# 📖 The Story: Building Homes Where the Streets Are Most Dangerous

### The question
New York is producing about **29,000 affordable homes a year**. Every new resident becomes a pedestrian, cyclist, bus rider or driver on the surrounding streets. **Is affordable housing being built where serious crashes happen, and does more housing mean more people getting badly hurt?** If the city's most affordable homes sit on its most dangerous streets, the families with the least money for cars or medical bills are the most exposed to traffic violence.

### The data
- **Affordable Housing Production by Building** (HPD, `hg8x-zxpr`): 9,551 buildings with affordable units, project start date, ZIP and coordinates.
- **SIRTA serious crashes** (`xeqp-qz8h`): 8,072 crashes (Oct 2022 – Jun 2026) with severe injuries, fatalities, ZIP and location.
- **Join:** the datasets share no key, and one building is not one crash, so Person 3 aggregated both to **ZIP code × year** and kept the **484 ZIP-years** present in both. 1,958 housing rows without a ZIP (only 1 % of units) could not be placed.

### What we found
1. **Housing and crashes go together across space.** ZIPs with more affordable units have more serious crashes (RQ1: Spearman ρ = 0.64), and ZIPs with more housing buildings have more severe injuries (RQ7: ρ = 0.73).
2. **The overlap is concentrated in two clusters** (RQ9 map): the South/Central **Bronx** (11 ZIPs) and Central/East **Brooklyn** (8 ZIPs). These 23 of 143 ZIPs (16 %) hold **44 % of affordable units, 34 % of serious crashes and 29 % of crash deaths**. 11207 (East New York) and 11212 (Brownsville) are the worst.
3. **Brooklyn leads in both housing and crashes** (RQ2: 33 % of units, 38 % of crashes). The **Bronx** builds almost as much housing with a much smaller crash share (24 %), and it has the **fewest crashes per housing project** (RQ8: 1.94 vs 5.53 in Staten Island).
4. **But more housing does not bring more injuries over time** (RQ10). Citywide production was flat (+0.6 %) from 2023 to 2025 while severe injuries rose 17.5 %. Within ZIPs, years with more new housing did not have more injuries (ρ = −0.09, not significant).
5. **Over the full years, crashes rose while housing did not** (RQ3, Youssef). In matched ZIP-years, 2023 → 2025 affordable units fell 2.7 % (28,832 → 28,059) while serious crashes rose 18.9 % (1,553 → 1,846).
6. **ZIPs with more housing projects have more severe injuries** (RQ4, Youssef): Pearson r = 0.71, Spearman ρ = 0.81 across 143 ZIPs (0.65 / 0.76 for the full years 2023–2025 only).
7. **High-in-both ZIP-years are a Bronx and Brooklyn pattern** (RQ5, Malak): of 316 ZIP-years in 2023–2025, 29 are in the top 25 % for both; 27 of them are in the Bronx (15) or Brooklyn (12). 11212 Brownsville in 2025 is the most extreme (1,090 units, 43 serious crashes).
8. **Some ZIPs have many crashes for little housing** (RQ6, Malak): 11234 (Brooklyn) and 11377 (Queens) have about 65 serious crashes per 100 affordable units in 2023–2025, so road safety there is a problem regardless of new housing.
9. *(RQ11 and RQ12 — Lama: add one line each when finished.)*

**What surprised us:** the link between housing and crashes is about **place**, not **construction**. Affordable housing is being built in neighbourhoods that were already dangerous. Building it doesn't seem to make them more dangerous, but nothing is making them safer either, as injuries keep rising.

### The limits
- **Correlation ≠ causation.** Dense, lower-income neighbourhoods with wide arterial roads have more of both housing and crashes. We cannot separate housing from these factors.
- **No population or traffic denominator.** Counts are not risk rates; big, busy ZIPs naturally have more crashes.
- **Partial years.** SIRTA starts in Oct 2022 and both datasets end in Jun 2026; year comparisons use 2023–2025 only, which is just 3 data points.
- **Timing.** Housing is dated by *project start*, and residents arrive years later. Effects of new residents could appear after our window.
- **Matched sample.** Only ZIP-years present in both datasets are integrated (484 of 1,500 groups); housing-only and crash-only ZIP-years are excluded. 16 % of crashes lack coordinates (still counted, not mapped).
- **ZIP codes are coarse.** They are postal routes, and a few span two boroughs (10463, 11237, 11385).
- **Housing includes preservation**, not just new buildings.

### The impact: who should act
- **NYC DOT / Vision Zero:** focus street redesigns (daylighting, protected bike lanes, leading pedestrian intervals, speed cameras) on the **23 overlap ZIPs**. One intervention there protects existing and future affordable-housing residents at once.
- **HPD & City Planning:** add a **street-safety check to site selection** for large affordable projects in overlap ZIPs and coordinate with DOT so safety work is done before residents move in.
- **Community boards & housing advocates in the Bronx and East Brooklyn:** use the map to ask for safety investment alongside new housing.
- **Next step for analysts:** add population and traffic volume to compute crash *rates*, and track the overlap ZIPs as projects are completed.

## Contribution — Person 5 (Kenzy)
| Area | Work |
|---|---|
| Research questions | RQ9 (overlap map), RQ10 (housing production vs severe injuries over time) |
| Visualizations | Interactive overlap map with toggleable raw crash/housing layers; indexed full-year trend; within-ZIP change scatter with correlation |
| Data work | ZIP centroids from building and crash coordinates; top-25 % overlap classification; full-year citywide totals; within-ZIP year-over-year changes |
| Consistency | Team colour and labelling rules, chart review checklist, fixes requested from teammates |
| Story | Question, data, findings (combining all team RQs), limitations and impact |
| Website | Wrote `app/charts/rq9_rq10.py` (RQ9 map + RQ10 chart, following the filters), `data/zip_centroids.csv`, and the story section text |